# P1-31 · Experiment E3: prespecified analysis of the 100 runs (CPU, ~10–20 min)
**Runtime: CPU.** Run after P1-28r (all 100 runs) and P1-27e (reader-2 scores). Read-only for the run folders.
Writes only `MyDrive/P1_v2/e3_analysis/`. Hospital results are written as aggregates only (no per-image hospital file is written here).

Analysis as stored before training (P1-28r, cell 1): primary contrasts HQB-UNet − HQB-cls, − HQB-trig, − HQB-free in mean per-image Dice
on the 596 VinDr images (out-of-fold), crossed hierarchical bootstrap over exams and seeds (10,000 replicates, `default_rng(0)`),
95% percentile interval, two-sided bootstrap P, Holm over the three contrasts; equivalence within ±0.01 if the 90% interval lies inside.
Secondary (external, no fine-tuning; per image and seed the Dice is averaged over the five fold models): mini-MIAS (women as clusters)
and the 84 hospital images (reader 1, reader 2 subset; patients as clusters), Holm within each reference set. Also: other region,
model levels, code replacement, finite shots, seed sensitivity, every fold and seed separately, and (added here, as reported
in the run log) the resumed runs: run metadata corrected from the correction files and a sensitivity analysis without resumed runs.

In [ ]:
# 1 · Drive, inputs
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, glob, json, numpy as np, pandas as pd
from scipy import stats
OUT = '/content/drive/MyDrive/P1_v2'; RUNS_DIR = f'{OUT}/e3v_runs'; AN = f'{OUT}/e3_analysis'; os.makedirs(AN, exist_ok=True)
B = 10000; PAIRS = (('hq', 'hc'), ('hq', 'ht'), ('hq', 'hf')); VAR = ('hq', 'hc', 'ht', 'hf')
runs = sorted(d for d in glob.glob(f'{RUNS_DIR}/E3v_*_seed*') if os.path.isdir(d))
assert len(runs) == 100, len(runs)
assert not any(os.path.exists(f'{d}/failed.json') for d in runs), 'a failed run exists'
RJ = pd.DataFrame([json.load(open(f'{d}/run.json')) for d in runs]); RJ['variant'] = RJ.name.str[-2:]
assert RJ.groupby(['variant', 'fold', 'seed']).size().eq(1).all() and len(RJ) == 100
REF = RJ.target.iloc[0]; OTHER = 'full' if REF == 'breast' else 'breast'; assert RJ.target.eq(REF).all()
print('runs 100 | target region:', REF, '| init check max', RJ.init_shared_max_abs_diff.abs().max(), '| switch check max', RJ.switch_check_max.max())
def load(f):
    X = pd.concat([pd.read_csv(f'{d}/{f}') for d in runs], ignore_index=True); X['variant'] = X.run.str[-2:]; return X
T = load('test_per_image.csv'); M = load('mias_per_image.csv'); H = load('hospital_per_image.csv')
assert T.groupby(['variant', 'seed']).file_id.nunique().eq(596).all() and not T.duplicated(['variant', 'seed', 'file_id']).any()
assert M.groupby('run').file_id.nunique().eq(322).all() and H.groupby('run').file_id.nunique().eq(84).all()
R2F = f'{OUT}/e3_reader2/hospital_reader2_per_image.csv.gz'
H2 = pd.read_csv(R2F) if os.path.exists(R2F) else None
print('reader 2 scores:', 'present' if H2 is not None else 'MISSING - run P1-27e first (reader-2 rows will be skipped)')
T['cluster'] = T.patient_id.astype(str)
M['cluster'] = M.file_id.str.extract(r'(\d+)')[0].astype(int).add(1).floordiv(2).astype(str)      # mini-MIAS: two images per woman
assert M.cluster.nunique() == 161
H['cluster'] = H.patient_id.astype(str)
RES = RJ.assign(resumed=RJ.resumed_at.apply(lambda v: len(v) > 0))[['name', 'fold', 'seed', 'resumed']]
print('resumed runs:', int(RES.resumed.sum()), RES[RES.resumed].name.str.cat(RES[RES.resumed].seed.astype(str), sep=' s').tolist())

In [ ]:
# 2 · Bootstrap (as in the Paper 1 analysis: crossed over clusters and seeds), Holm
def boot(u, rng=None):
    rng = rng or np.random.default_rng(0)
    seeds = np.sort(u.seed.unique()); cl = np.sort(u.cluster.unique())
    ci = {c: i for i, c in enumerate(cl)}; si = {s: i for i, s in enumerate(seeds)}
    S = np.zeros((len(cl), len(seeds))); N = np.zeros((len(cl), len(seeds)))
    np.add.at(S, (u.cluster.map(ci).values, u.seed.map(si).values), u.v.values)
    np.add.at(N, (u.cluster.map(ci).values, u.seed.map(si).values), 1)
    est = S.sum() / N.sum()
    wc = rng.multinomial(len(cl), np.ones(len(cl)) / len(cl), size=B)
    ws = rng.multinomial(len(seeds), np.ones(len(seeds)) / len(seeds), size=B)
    bs = np.einsum('bc,cs,bs->b', wc, S, ws) / np.einsum('bc,cs,bs->b', wc, N, ws)
    p = max(min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean())), 1.0 / B)
    return dict(est=est, lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5), lo90=np.percentile(bs, 5), hi90=np.percentile(bs, 95), p=p,
                n_clusters=len(cl), n_seeds=len(seeds), n_images=int(N[:, 0].sum()))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for r, i in enumerate(o):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1, run)
    return adj
def pair(X, a, b, col, keys):
    x = X[X.variant == a][keys + ['cluster', col]].rename(columns={col: 'a'})
    y = X[X.variant == b][keys + ['cluster', col]].rename(columns={col: 'b'})
    m = x.merge(y, on=keys + ['cluster']); assert len(m) == len(x) == len(y), (a, b, col)
    return m.assign(v=m.a - m.b).dropna(subset=['v'])
def ext(X, col):
    """External sets: per image and seed, Dice averaged over the five fold models."""
    g = X.dropna(subset=[col]).groupby(['variant', 'seed', 'file_id', 'cluster'], as_index=False)[col].mean()
    assert g.groupby(['variant', 'seed', 'file_id']).size().eq(1).all()
    return g
print('ok')

In [ ]:
# 3 · Contrasts: primary (VinDr out-of-fold) and secondary (mini-MIAS; hospital reader 1, reader 2, rule-based proxy)
SETS = [('vindr_test', T, f'dice', ['seed', 'file_id'], False)]
for col in (f'dice_{REF}', f'dice_{OTHER}'): SETS.append(('mini_mias', M, col, ['seed', 'file_id'], True))
for col in (f'dice_reader1_{REF}', f'dice_reader1_{OTHER}', 'dice_proxy'): SETS.append(('hospital', H, col, ['seed', 'file_id'], True))
if H2 is not None:
    H2['cluster'] = H2.patient_id.astype(str)
    for col in (f'dice_reader2_{REF}', f'dice_reader2_{OTHER}'): SETS.append(('hospital_reader2', H2, col, ['seed', 'file_id'], True))
rows = []
for sname, X, col, keys, is_ext in SETS:
    Y = ext(X, col) if is_ext else X
    for a, b in PAIRS:
        rows.append(dict(set=sname, metric=col, contrast=f'{a}-{b}', **boot(pair(Y, a, b, col, keys))))
C = pd.DataFrame(rows)
C['region'] = np.where(C.metric.str.endswith(REF) | (C.metric == 'dice'), 'target', np.where(C.metric == 'dice_proxy', 'proxy', 'other'))
C['holm'] = np.nan
for (s, m), g in C.groupby(['set', 'metric']): C.loc[g.index, 'holm'] = holm(g.p)
C['resolved'] = C.holm < 0.05
C['equivalent_0.01'] = (C.lo90 > -0.01) & (C.hi90 < 0.01)
C.to_csv(f'{AN}/contrasts.csv', index=False)
pd.set_option('display.width', 250)
print(C[['set', 'metric', 'contrast', 'est', 'lo', 'hi', 'p', 'holm', 'resolved', 'equivalent_0.01', 'n_clusters', 'n_images']].round(5).to_string())

In [ ]:
# 4 · Model levels, code replacement, finite shots
lv = []
for sname, X, col, keys, is_ext in SETS:
    Y = ext(X, col) if is_ext else X
    for v in VAR:
        lv.append(dict(set=sname, metric=col, variant=v, **boot(Y[Y.variant == v].rename(columns={col: 'v'})[['seed', 'cluster', 'v']].dropna())))
for col in ('drop_const', 'drop_swapavg'):
    for v in VAR:
        lv.append(dict(set='vindr_test', metric=col, variant=v, **boot(T[T.variant == v].rename(columns={col: 'v'})[['seed', 'cluster', 'v']].dropna())))
L = pd.DataFrame(lv); L.to_csv(f'{AN}/levels.csv', index=False)
print(L[['set', 'metric', 'variant', 'est', 'lo', 'hi']].round(4).to_string())
sh = []
for n in (100, 1000, 10000):
    t = T[T.variant == 'hq'].copy(); t['v'] = t[f'dice_shots{n}'] - t.dice
    sh.append(dict(shots=n, **boot(t[['seed', 'cluster', 'v']])))
SH = pd.DataFrame(sh); SH.to_csv(f'{AN}/shots.csv', index=False); print(SH.round(5).to_string())

In [ ]:
# 5 · Seed sensitivity and every fold / seed separately (primary set)
ss, fs = [], []
for a, b in PAIRS:
    m = pair(T, a, b, 'dice', ['seed', 'file_id'])
    d = m.groupby('seed').v.mean(); n = len(d); tq = stats.t.ppf(0.975, n - 1)
    ss.append(dict(contrast=f'{a}-{b}', method='t interval over seed-level differences', est=d.mean(),
                   lo=d.mean() - tq * d.std(ddof=1) / np.sqrt(n), hi=d.mean() + tq * d.std(ddof=1) / np.sqrt(n)))
    rng = np.random.default_rng(0); bs = np.array([d.values[rng.integers(0, n, n)].mean() for _ in range(B)])
    ss.append(dict(contrast=f'{a}-{b}', method='seed resampling only', est=d.mean(), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5)))
    for s in sorted(m.seed.unique()):
        r = boot(m[m.seed != s]); ss.append(dict(contrast=f'{a}-{b}', method=f'leave out seed {s}', est=r['est'], lo=r['lo'], hi=r['hi'], p=r['p']))
    mf = m.merge(T[T.variant == a][['seed', 'file_id', 'fold']], on=['seed', 'file_id'])
    for (f_, s_), g in mf.groupby(['fold', 'seed']): fs.append(dict(contrast=f'{a}-{b}', fold=f_, seed=s_, est=g.v.mean(), n=len(g)))
    for s_, g in mf.groupby('seed'): fs.append(dict(contrast=f'{a}-{b}', fold='all', seed=s_, est=g.v.mean(), n=len(g)))
    for f_, g in mf.groupby('fold'): fs.append(dict(contrast=f'{a}-{b}', fold=f_, seed='all', est=g.v.mean(), n=len(g)))
SS = pd.DataFrame(ss); SS.to_csv(f'{AN}/seed_sensitivity.csv', index=False); print(SS.round(5).to_string())
FS = pd.DataFrame(fs); FS.to_csv(f'{AN}/by_fold_seed.csv', index=False)
print(FS[(FS.fold == 'all') | (FS.seed == 'all')].round(5).to_string())

In [ ]:
# 6 · Resumed runs: corrected run metadata (from history.csv and the correction files) and primary contrasts without resumed runs
cf = sorted(glob.glob(f'{OUT}/e3v_run_corrections*.csv'))
CORR = pd.concat([pd.read_csv(f) for f in cf], ignore_index=True).drop_duplicates(['run', 'seed', 'field']) if cf else pd.DataFrame()
print('correction files:', [os.path.basename(f) for f in cf]); print(CORR[['run', 'seed', 'field', 'value_in_run_json', 'corrected_value']].to_string() if len(CORR) else '-')
meta = RJ[['name', 'variant', 'fold', 'seed', 'epochs_run', 'best_epoch', 'final_lr', 'minutes', 'resumed_at', 'session', 'tf_version']].copy()
for _, c in CORR.iterrows():
    k = (meta.name == c.run) & (meta.seed == c.seed); assert k.sum() == 1, (c.run, c.seed)
    assert float(meta.loc[k, c.field].iloc[0]) == float(c.value_in_run_json), (c.run, c.seed, c.field)
    meta.loc[k, c.field] = c.corrected_value
meta['corrected'] = meta.name.str.cat(meta.seed.astype(str)).isin(CORR.run.str.cat(CORR.seed.astype(str))) if len(CORR) else False
meta.to_csv(f'{AN}/run_metadata_corrected.csv', index=False)
print(meta.groupby('variant')[['epochs_run', 'best_epoch', 'minutes']].mean().round(2).to_string())
rs = []
bad = set(map(tuple, RES[RES.resumed][['fold', 'seed']].values))       # a fold-seed cell is dropped for every variant if any of its runs resumed
keep = ~T[['fold', 'seed']].apply(tuple, axis=1).isin(bad)
for a, b in PAIRS:
    m = pair(T[keep], a, b, 'dice', ['seed', 'file_id'])
    rs.append(dict(contrast=f'{a}-{b}', analysis='without fold-seed cells containing a resumed run', cells_dropped=len(bad), **boot(m)))
RS = pd.DataFrame(rs); RS.to_csv(f'{AN}/resumed_sensitivity.csv', index=False); print(RS.round(5).to_string())

In [ ]:
# 7 · Public-data per-image tables for the figures (VinDr out-of-fold, mini-MIAS); no hospital per-image file leaves this folder
T.drop(columns=['cluster']).to_csv(f'{AN}/e3_vindr_test_per_image.csv.gz', index=False)
M.drop(columns=['cluster']).to_csv(f'{AN}/e3_mias_per_image.csv.gz', index=False)
with open(f'{AN}/README.txt', 'w') as f:
    f.write('P1-31 E3 analysis outputs. contrasts.csv (primary: set=vindr_test), levels.csv, shots.csv, seed_sensitivity.csv, by_fold_seed.csv, '
            'run_metadata_corrected.csv, resumed_sensitivity.csv; per-image tables for VinDr and mini-MIAS only. Hospital results: aggregates only.\n')
print(sorted(os.listdir(AN))); print('DONE')